# 9.11 Full fine-tuning vs parameter-efficient LoRA adapters

LoRA and related methods adapt a large model by learning small side parameters instead of moving every weight.

This notebook implements the real low-rank update $W'=W+BA$ on small NumPy classifiers. The D5 rung makes the rank bottleneck visible without training or downloading a model.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 911
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)


def build_lora_ladder():
    rng = np.random.default_rng(SEED)
    ladders = []
    X_d1 = np.array([[1.0, -0.5, 0.2]])
    W_base_d1 = np.array([[0.5, -0.2], [0.1, 0.3], [-0.4, 0.2]])
    delta_d1 = np.array([[0.2, -0.1], [0.0, 0.1], [0.1, 0.0]])
    y_d1 = np.argmax(X_d1 @ (W_base_d1 + delta_d1), axis=1)
    ladders.append({"name": "D1 one prompt", "X": X_d1, "W": W_base_d1, "delta": delta_d1, "y": y_d1})

    X_d2 = rng.normal(size=(20, 6))
    W_d2 = rng.normal(scale=0.35, size=(6, 3))
    left_d2 = rng.normal(scale=0.30, size=(6, 2))
    right_d2 = rng.normal(scale=0.30, size=(2, 3))
    delta_d2 = left_d2 @ right_d2
    y_d2 = np.argmax(X_d2 @ (W_d2 + delta_d2), axis=1)
    ladders.append({"name": "D2 few-shot domain", "X": X_d2, "W": W_d2, "delta": delta_d2, "y": y_d2})

    X_d3 = rng.normal(size=(36, 8))
    W_d3 = rng.normal(scale=0.30, size=(8, 4))
    delta_d3 = rng.normal(scale=0.22, size=(8, 4))
    delta_d3[:, 0] += np.linspace(-0.8, 0.8, 8)
    y_d3 = np.argmax(X_d3 @ (W_d3 + delta_d3), axis=1)
    ladders.append({"name": "D3 higher-rank distractors", "X": X_d3, "W": W_d3, "delta": delta_d3, "y": y_d3})

    X_d4 = np.array([
        [1, 1, 0, 0, 0, 1],
        [1, 0, 1, 0, 1, 0],
        [0, 1, 0, 1, 0, 1],
        [0, 0, 1, 1, 1, 0],
        [1, 1, 1, 0, 0, 0],
        [0, 1, 1, 0, 1, 0],
        [1, 0, 0, 1, 0, 1],
        [0, 0, 0, 1, 1, 1]
    ], dtype=float)
    W_d4 = rng.normal(scale=0.25, size=(6, 3))
    delta_d4 = rng.normal(scale=0.18, size=(6, 3))
    y_d4 = np.argmax(X_d4 @ (W_d4 + delta_d4), axis=1)
    ladders.append({"name": "D4 text corpus", "X": X_d4, "W": W_d4, "delta": delta_d4, "y": y_d4})

    X_d5 = rng.normal(size=(90, 12))
    W_d5 = rng.normal(scale=0.18, size=(12, 5))
    U = rng.normal(size=(12, 5))
    _, singular_values, Vt = np.linalg.svd(U, full_matrices=False)
    shaped_singular_values = np.array([1.8, 1.3, 0.9, 0.6, 0.4])
    delta_d5 = U @ np.diag(shaped_singular_values / (singular_values + 1e-9)) @ Vt
    y_d5 = np.argmax(X_d5 @ (W_d5 + delta_d5), axis=1)
    ladders.append({"name": "D5 rank bottleneck", "X": X_d5, "W": W_d5, "delta": delta_d5, "y": y_d5})
    return ladders


def low_rank_approx(matrix, rank):
    U, singular_values, Vt = np.linalg.svd(matrix, full_matrices=False)
    return U[:, :rank] @ np.diag(singular_values[:rank]) @ Vt[:rank, :]


def classifier_accuracy(X, W, labels):
    predictions = np.argmax(X @ W, axis=1)
    return float(np.mean(predictions == labels))


## The concept, built once (D1)

LoRA freezes $W$ and applies $$W'=W+\Delta W,\quad \Delta W=BA,\; B\in\mathbb{R}^{d\times r},\; A\in\mathbb{R}^{r\times k}.$$
The lesson counts are exact: a full $64\times64$ matrix has $4096$ weights; rank $4$ LoRA has $64\times4+4\times64=512$ weights; the fraction is $12.5\%$; an adapter bottleneck $8$ has $1024$ weights; and 10 prefix tokens of width 64 have 640 parameters per layer.

In [ ]:

def apply_lora_update(W, B, A, alpha=1.0):
    delta = (alpha / B.shape[1]) * (B @ A)
    updated = W + delta
    return updated, delta

full_count = 64 * 64
lora_count = 64 * 4 + 4 * 64
lora_fraction = lora_count / full_count
adapter_count = 64 * 8 + 8 * 64
prefix_count = 10 * 64
assert full_count == 4096
assert lora_count == 512
assert round(lora_fraction * 100, 1) == 12.5
assert adapter_count == 1024
assert prefix_count == 640
print(full_count, lora_count, lora_fraction, adapter_count, prefix_count)


Now apply a genuine low-rank update to a toy classifier and compare logits before and after merging $BA$ into $W$.

In [ ]:

W_demo = np.array([[0.5, -0.2], [0.1, 0.3], [-0.4, 0.2]])
B_demo = np.array([[0.3], [0.1], [-0.2]])
A_demo = np.array([[0.4, -0.5]])
W_updated, delta_demo = apply_lora_update(W_demo, B_demo, A_demo, alpha=1.0)
x_demo = np.array([[1.0, -0.5, 0.2]])
base_logits = x_demo @ W_demo
updated_logits = x_demo @ W_updated
assert delta_demo.shape == W_demo.shape
print("delta", delta_demo)
print("base logits", base_logits)
print("updated logits", updated_logits)


## The dataset ladder

Each rung supplies a base classifier and a target domain update. The notebook approximates that target update with a chosen LoRA rank.

In [ ]:

ladder = build_lora_ladder()
for rung in ladder:
    print(rung["name"], "X", rung["X"].shape, "W", rung["W"].shape, "classes", np.bincount(rung["y"], minlength=rung["W"].shape[1]))
    print("delta rank", np.linalg.matrix_rank(rung["delta"]))


## Run the same LoRA approximation across D1-D5

In [ ]:

results = []
for index, rung in enumerate(ladder, start=1):
    rank = min(2, min(rung["delta"].shape))
    delta_low_rank = low_rank_approx(rung["delta"], rank)
    W_lora = rung["W"] + delta_low_rank
    accuracy = classifier_accuracy(rung["X"], W_lora, rung["y"])
    trainable = rung["W"].shape[0] * rank + rank * rung["W"].shape[1]
    metric = accuracy / max(trainable, 1)
    results.append({"name": rung["name"], "rank": rank, "delta": delta_low_rank, "target_delta": rung["delta"], "accuracy": accuracy, "metric": metric, "trainable": trainable})

print("rung | rank | accuracy | trainable | accuracy_per_parameter")
for result in results:
    print(f"{result['name']} | {result['rank']} | {result['accuracy']:.3f} | {result['trainable']} | {result['metric']:.5f}")


## Results visualization

Small multiples show learned update matrices. The summary curve compares accuracy against trainable parameter counts.

In [ ]:

fig, axes = plt.subplots(1, len(results), figsize=(16, 3))
for ax, result in zip(axes, results):
    image = ax.imshow(result["delta"], aspect="auto", cmap="coolwarm")
    ax.set_title(result["name"].split()[0])
    ax.set_xlabel("class")
    ax.set_ylabel("feature")
fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.7)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([item["trainable"] for item in results], [item["accuracy"] for item in results], marker="o")
ax.set_xlabel("trainable parameters")
ax.set_ylabel("accuracy")
ax.set_title("Quality vs PEFT budget")
plt.show()


## Pitfall on the hardest rung

Pitfall: rank too small bottlenecks $BA$. On D5, compare rank 1, rank 2, and a full-update baseline.

In [ ]:

d5 = ladder[-1]
for rank in [1, 2, 4, min(d5["delta"].shape)]:
    delta_rank = low_rank_approx(d5["delta"], rank)
    W_rank = d5["W"] + delta_rank
    accuracy = classifier_accuracy(d5["X"], W_rank, d5["y"])
    residual = float(np.linalg.norm(d5["delta"] - delta_rank))
    print("rank", rank, "accuracy", accuracy, "residual", residual)
full_accuracy = classifier_accuracy(d5["X"], d5["W"] + d5["delta"], d5["y"])
print("full tune baseline", full_accuracy)


## Evaluate it + Practice

- Metric: accuracy per trainable parameter, with raw accuracy as a quality floor.
- No-skill baseline: full fine-tuning moves every entry of $W$.
- Cheap sanity check: rank-4 at width 64 must have 512 trainable parameters.
- Ablation: set $BA=0$ and confirm domain accuracy falls back to the frozen base.
- Failure signals: stable-looking low-rank residual with poor D5 accuracy or forgotten adapter merge at inference.

Practice prompts:


**Practice.** Raise the D5 rank and explain the residual norm change.

**Practice.** Change alpha in `apply_lora_update` and inspect logits.

**Practice.** Compare adapter and prefix parameter counts for width 128.